In [1]:
import os
import time
import pandas as pd
import requests
from dotenv import load_dotenv

In [2]:
# 1. Load the input spreadsheet (columns: Code, English, Recraft Prompt)
df = pd.read_csv('midjourney_prompts.csv')
df['English'] = df['English'].str.strip()
print(len(df), "rows")

137 rows


In [3]:
# 2. API setup. RECRAFT_API_KEY (and optional RECRAFT_STYLE_ID) are read from the .env file.
load_dotenv()
API_KEY = os.environ["RECRAFT_API_KEY"]
STYLE_ID = os.environ.get("RECRAFT_STYLE_ID") or None  # custom style made with recraftv3; optional
MODEL = "recraftv3"                                    # Recraft V3, raster output
STYLE = "digital_illustration"                        # curated V3 style; ignored if STYLE_ID is set
URL = "https://external.api.recraft.ai/v1/images/generations"
BG_URL = "https://external.api.recraft.ai/v1/images/removeBackground"  # ~$0.01 per image
headers = {"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"}

In [4]:
# 3. Generation + download helpers
IMG_DIR = 'generated/png_v3'      # separate folder from the earlier runs
os.makedirs(IMG_DIR, exist_ok=True)

def img_path(code, word):
    safe = "".join(c if c.isalnum() else "_" for c in word)
    return f"{IMG_DIR}/{code}_{safe}.png"

def generate_recraft_asset(code, word, prompt):
    path = img_path(code, word)
    if os.path.exists(path):          # resume support
        return path
    payload = {
        "prompt": prompt,             # per-word descriptive prompt from the CSV
        "model": MODEL,
        "size": "1:1",
        "image_format": "png",
        "response_format": "url",
    }
    if STYLE_ID:                      # custom style overrides the curated one
        payload["style_id"] = STYLE_ID
    else:
        payload["style"] = STYLE
    for attempt in range(3):
        r = requests.post(URL, json=payload, headers=headers, timeout=120)
        if r.status_code == 429:
            time.sleep(5 * (attempt + 1))
            continue
        if r.status_code != 200:
            print(f"[{code} {word}] HTTP {r.status_code}: {r.text[:200]}")
            return None
        img_url = r.json()["data"][0]["url"]   # temporary URL (~24h): download now
        # Make the background transparent (separate API call), then download that result
        bg = requests.post(BG_URL, json={"image_url": img_url, "image_format": "png"},
                           headers=headers, timeout=120)
        if bg.status_code != 200:
            print(f"[{code} {word}] remove-background HTTP {bg.status_code}: {bg.text[:200]}")
            return None
        img = requests.get(bg.json()["image"]["url"], timeout=120)
        img.raise_for_status()
        with open(path, "wb") as f:
            f.write(img.content)
        return path
    return None

In [5]:
# 4. Test on a few rows first to check the look before spending credits on all of them
TEST_N = 5
for _, row in df.head(TEST_N).iterrows():
    print(row['Code'], row['English'], '->', generate_recraft_asset(row['Code'], row['English'], row['Recraft Prompt']))

101 Noise -> generated/png_v3/101_Noise.png
102 Laughter -> generated/png_v3/102_Laughter.png
103 Wrestling -> generated/png_v3/103_Wrestling.png
104 Dance -> generated/png_v3/104_Dance.png
105 Work -> generated/png_v3/105_Work.png


In [6]:
# 5. Full batch (skips files that already exist, so it is safe to re-run)
paths = []
for _, row in df.iterrows():
    paths.append(generate_recraft_asset(row['Code'], row['English'], row['Recraft Prompt']))
    time.sleep(1)
df['Image_Path'] = paths
df['Status'] = df['Image_Path'].apply(lambda p: 'ok' if p else 'failed')

os.makedirs('generated', exist_ok=True)
df.to_csv('generated/recraft_bulk_results.csv', index=False)
print(df['Status'].value_counts())

Status
ok    137
Name: count, dtype: int64
